# ObsPy Waveform Demo

This notebook demonstrates loading and plotting seismic waveforms using ObsPy.  
Data is fetched live from the IRIS FDSN web service, so an internet connection is required for the download cells.

## 1 – Built-in example data (no download needed)

In [ ]:
import obspy
print('ObsPy version:', obspy.__version__)

In [ ]:
# Load one of ObsPy's bundled example waveforms
from obspy import read

st = read()   # loads 3-component example MiniSEED data
print(st)

In [ ]:
# Basic waveform plot – rendered inline by Jupyter
import matplotlib
matplotlib.use('Agg')       # non-interactive backend; safe inside Docker
import matplotlib.pyplot as plt

fig = st.plot(show=False)   # returns a matplotlib Figure
plt.savefig('/workspace/data/example_waveform.png', dpi=150, bbox_inches='tight')
plt.show()                  # renders inline in JupyterLab
print('Plot also saved to /workspace/data/example_waveform.png')

## 2 – Inspect a single trace

In [ ]:
tr = st[0]
print('Network  :', tr.stats.network)
print('Station  :', tr.stats.station)
print('Channel  :', tr.stats.channel)
print('Sampling :', tr.stats.sampling_rate, 'Hz')
print('Start    :', tr.stats.starttime)
print('End      :', tr.stats.endtime)
print('Samples  :', tr.stats.npts)

## 3 – Bandpass filter and compare

In [ ]:
from obspy import read
import matplotlib.pyplot as plt

raw = read()
filtered = raw.copy().filter('bandpass', freqmin=1.0, freqmax=10.0)

fig, axes = plt.subplots(3, 2, figsize=(14, 8), sharex=True)
fig.suptitle('Raw (left) vs Bandpass 1–10 Hz (right)')

for i, (tr_raw, tr_filt) in enumerate(zip(raw, filtered)):
    t = tr_raw.times()
    axes[i, 0].plot(t, tr_raw.data, lw=0.7, color='steelblue')
    axes[i, 0].set_ylabel(tr_raw.stats.channel)
    axes[i, 1].plot(t, tr_filt.data, lw=0.7, color='tomato')

axes[-1, 0].set_xlabel('Time (s)')
axes[-1, 1].set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig('/workspace/data/filtered_waveform.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to /workspace/data/filtered_waveform.png')

## 4 – Spectrogram

In [ ]:
from obspy import read

tr = read()[0]
fig = tr.spectrogram(show=False, log=True)
fig.savefig('/workspace/data/spectrogram.png', dpi=150, bbox_inches='tight')
fig.show()
print('Saved to /workspace/data/spectrogram.png')

## 5 – Download real data from IRIS FDSN (requires internet)

In [ ]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime

client = Client('IRIS')

# 2011 Tohoku earthquake – adjust times/network/station as needed
t_start = UTCDateTime('2011-03-11T05:46:23')
t_end   = t_start + 600   # 10 minutes

st_real = client.get_waveforms(
    network='IU', station='ANMO', location='00', channel='BH*',
    starttime=t_start, endtime=t_end
)

st_real.detrend('demean')
st_real.taper(max_percentage=0.05)
print(st_real)

fig = st_real.plot(show=False)
fig.savefig('/workspace/data/tohoku_IU_ANMO.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to /workspace/data/tohoku_IU_ANMO.png')

## 6 – Read your own file

Drop a SAC / MiniSEED / SEG-Y / etc. file into the `data/` folder on your host,  
then load it here:

In [ ]:
# Uncomment and edit the path to load your own waveform file
# from obspy import read
# st_my = read('/workspace/data/my_waveform.mseed')
# st_my.plot()